# Tutorial 6 — Chemical Space Visualization with UMAP
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Chemical space is the universe of all possible molecules. Visualizing where compounds cluster reveals SAR trends, diversity gaps, and potential design opportunities. We use Morgan fingerprints + UMAP dimensionality reduction.

In [ ]:
!pip install rdkit umap-learn scikit-learn pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Compound library with class labels
compounds = [
    # NSAIDs
    ("Aspirin",       "CC(=O)Oc1ccccc1C(=O)O",              "NSAID"),
    ("Ibuprofen",     "CC(C)Cc1ccc(cc1)C(C)C(=O)O",         "NSAID"),
    ("Naproxen",      "COc1ccc2cc(C(C)C(=O)O)ccc2c1",       "NSAID"),
    ("Celecoxib",     "Cc1ccc(-c2cc(NS(=O)(=O)c3ccc(N)cc3)no2)cc1", "NSAID"),
    ("Diclofenac",    "OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl",     "NSAID"),
    # Opioids
    ("Morphine",      "OC1=CC=C2CC3N(CCC34CCc5c4cc(O)c(OC)c5)C2=C1", "Opioid"),
    ("Codeine",       "COc1ccc2CC3N(CCC34CCc5c4cc(OC)c(O)c5)C2=c1",  "Opioid"),
    ("Tramadol",      "OC1(c2ccccc2)CCCCC1CN(C)C",           "Opioid"),
    # Stimulants
    ("Caffeine",      "Cn1cnc2c1c(=O)n(C)c(=O)n2C",         "Stimulant"),
    ("Amphetamine",   "CC(N)Cc1ccccc1",                      "Stimulant"),
    ("Modafinil",     "NC(=O)CS(=O)c1ccc(-c2ccccc2)cc1",    "Stimulant"),
    # Antidepressants
    ("Fluoxetine",    "CNCCC(c1ccccc1)Oc1ccc(cc1)C(F)(F)F", "Antidepressant"),
    ("Sertraline",    "CNC1CCC(c2ccc(Cl)c(Cl)c2)c2ccccc21", "Antidepressant"),
    ("Venlafaxine",   "COc1ccc(C2(CCN(C)C)CCCCC2)cc1",      "Antidepressant"),
    # Antibiotics
    ("Ciprofloxacin", "OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O", "Antibiotic"),
    ("Amoxicillin",   "CC1(C)SC2C(NC(=O)C(N)c3ccc(O)cc3)C(=O)N2C1C(=O)O", "Antibiotic"),
    ("Azithromycin",  "CC(CC1OC(=O)CC1CC)C1CC(C)C(=O)C(OC2CC(CC(O2)C)N(C)C)C(C)CC(CC(OC3OC(C)CC(O)C3OC)C(C)CCC(=O)O1)OC", "Antibiotic"),
]

mols  = [(n, Chem.MolFromSmiles(s), c) for n, s, c in compounds if Chem.MolFromSmiles(s)]
fps   = np.array([AllChem.GetMorganFingerprintAsBitVect(m, 2, 1024) for _, m, _ in mols])
names = [n for n, _, _ in mols]
cats  = [c for _, _, c in mols]
print(f"Featurized {len(mols)} molecules")

In [ ]:
# PCA
pca    = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(fps)

# UMAP
reducer = UMAP(n_components=2, n_neighbors=5, min_dist=0.3, random_state=42)
coords_umap = reducer.fit_transform(fps)

palette = {
    "NSAID": "#1565c0", "Opioid": "#b71c1c",
    "Stimulant": "#e65100", "Antidepressant": "#7b1fa2",
    "Antibiotic": "#2e7d32",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, coords, title in [(ax1, coords_pca, "PCA"), (ax2, coords_umap, "UMAP")]:
    for cat, color in palette.items():
        mask = [c == cat for c in cats]
        x = coords[mask, 0]; y = coords[mask, 1]
        ax.scatter(x, y, c=color, label=cat, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
        for xi, yi, n in zip(x, y, [names[i] for i, m in enumerate(mask) if m]):
            ax.annotate(n, (xi, yi), fontsize=7, ha='center', va='bottom',
                        xytext=(0, 4), textcoords='offset points', color=color)
    ax.set_title(f"Chemical Space — {title}", fontsize=12)
    ax.legend(fontsize=9, framealpha=0.8)
    ax.set_xlabel(f"{title}1"); ax.set_ylabel(f"{title}2")

plt.tight_layout()
plt.savefig("chemical_space.png", dpi=150)
plt.show()
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

## Key takeaways
- UMAP preserves local structure better than PCA for non-linear chemical space
- Compounds cluster by drug class even without class labels — fingerprints capture pharmacophoric features
- Gaps in chemical space are opportunities for novel scaffold design
- Use t-SNE or UMAP for exploratory analysis; PCA for interpretability